# VantaDB Quickstart

**Embedded Rust engine for durable local memory and hybrid vector retrieval.**

This notebook walks through the `vantadb-py` Python SDK end-to-end: install, CRUD, vector search, hybrid search (BM25 + cosine via RRF), and metadata filters. No server or network required — everything runs in-process.

## 1. Install

The distribution name is `vantadb-py`; the importable module is `vantadb_py`.

In [ ]:
%%capture
!pip install vantadb-py

## 2. Open (or create) an embedded database

`VantaDB(db_path, ...)` opens a persistent local store. Zero configuration.

In [ ]:
import vantadb_py as vanta

db = vanta.VantaDB("./vantadb_colab", memory_limit_bytes=512_000_000)
print("db open")

## 3. Insert records with vectors, payload, and metadata

Use `put()` to store a namespace-scoped record. Compatible vectors: a Python `List[float]`.

In [ ]:
db.put(
    namespace="agent/main",
    key="memory-1",
    payload="The user prefers dark mode in all applications.",
    metadata={"category": "ui", "priority": 1},
    vector=[0.12, 0.88, 0.54],
)
db.put(
    namespace="agent/main",
    key="memory-2",
    payload="Vector databases store embeddings and retrieve them by similarity.",
    metadata={"category": "db", "priority": 2},
    vector=[0.91, 0.22, 0.36],
)
db.put(
    namespace="agent/main",
    key="memory-3",
    payload="A local-first engine avoids network calls for latency-sensitive agents.",
    metadata={"category": "db", "priority": 3},
    vector=[0.88, 0.31, 0.35],
)
print("3 records inserted")

## 4. Read a record by key

In [ ]:
rec = db.get_memory("agent/main", "memory-1")
print(rec.payload)
print("metadata:", rec.metadata)

## 5. Hybrid search (BM25 + vector, fused via RRF)

Provide a `text_query` for lexical scoring and a `query_vector` for vector similarity.

In [ ]:
hits = db.search_memory(
    namespace="agent/main",
    text_query="local first latency",
    query_vector=[0.85, 0.35, 0.40],
    top_k=3,
)
for h in hits:
    print(f"{h.key}  score={h.score:.3f}  {h.payload}")

## 6. Metadata filters

Filter by scalar metadata in both `list_memory()` and `search_memory()`.

In [ ]:
# List records filtered by metadata (no query)
res = db.list_memory("agent/main", filters={"category": "db"}, limit=10)
for r in res.records:
    print("list filter ->", r.key, r.metadata)

In [ ]:
# Hybrid search + metadata filter
hits = db.search_memory(
    namespace="agent/main",
    query_vector=[0.9, 0.2, 0.4],
    filters={"category": "db"},
    top_k=3,
)
print(f"{len(hits)} hits within category 'db':")
for h in hits:
    print(" ", h.key, h.metadata)

## 7. Update / delete

`put()` with an existing key updates. `delete_memory()` removes a record.

In [ ]:
db.put(
    namespace="agent/main",
    key="memory-1",
    payload="The user prefers dark mode everywhere, including editors and terminals.",
    metadata={"category": "preference", "priority": 1},
    vector=[0.12, 0.88, 0.54],
)
updated = db.get_memory("agent/main", "memory-1")
print("updated:", updated.payload)

deleted = db.delete_memory("agent/main", "memory-2")
print("deleted memory-2:", deleted)
print("get deleted ->", db.get_memory("agent/main", "memory-2"))

## 8. Durability & shutdown

`flush()` writes the WAL and HNSW index to disk; `close()` releases the engine.

In [ ]:
db.flush()
db.close()
print("done. Database persisted locally.")

## Next steps

- Docs: [docs/api/PYTHON_SDK.md](https://github.com/ness-e/Vantadb/blob/main/docs/api/PYTHON_SDK.md)
- Repo: [github.com/ness-e/Vantadb](https://github.com/ness-e/Vantadb)